# Notebook 02: Data Cleaning & Engineering

This notebook represents the robust, production-ready pipeline for preparing raw banking data for Exploratory Data Analysis (EDA) and subsequent Machine Learning models.

## Pipeline Objectives
- **Data Integrity:** Gracefully handle missing inputs and anomalies logically without unnecessary row destruction.
- **Feature Augmentation:** Extract sophisticated temporal and monetary behaviors dynamically.
- **Reliability:** Implement defensive programming and strict validation constraints ensuring crash-free, consistent output.
- **Traceability:** High-visibility structured logging for stakeholder interpretability.

### Step 1: Initialization and Data Loading
We load libraries and establish robust pathing. Strict logging is initiated.

In [2]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("Starting Data Cleaning & Feature Engineering Pipeline...")

# Load the dataset safely
raw_data_path = '../data/raw/Banking_churn_prediction.csv'
if not os.path.exists(raw_data_path):
    raise FileNotFoundError(f"CRITICAL: Raw data file missing at {raw_data_path}")

df = pd.read_csv(raw_data_path)
original_shape = df.shape
original_cols = set(df.columns)
print(f"Initial data loaded. Size: {original_shape[0]} rows, {original_shape[1]} columns.")

Starting Data Cleaning & Feature Engineering Pipeline...
Initial data loaded. Size: 28382 rows, 21 columns.


### Step 2: Column Normalization
Standardizing column names prevents index mismatches during production scoring endpoints.

In [3]:
# Standardize column names to snake_case safely
df.columns = df.columns.astype(str).str.strip().str.replace(' ', '_', regex=False).str.lower()
print("Columns standardized.")

Columns standardized.


### Step 3: Duplicate Handling
Identical signals bias ML convergence. We scan and clean these prior to imputation.

In [4]:
# Explicitly compute duplicates
duplicate_count = df.duplicated().sum()
print(f"Duplicate row scan: {duplicate_count} found.")

if duplicate_count > 0:
    df.drop_duplicates(inplace=True)
    print(f"{duplicate_count} Duplicates removed successfully.")
else:
    print("No duplicates detected.")

Duplicate row scan: 0 found.
No duplicates detected.


### Step 4: Datetime Parsing (`last_transaction`)
We safely coerce temporal structures.
> **Business Context:** Missing values (`NaT`) natively exist here indicating account stagnation. They are captured and retained actively as signals rather than imputed.

In [5]:
if 'last_transaction' in df.columns:
    df['last_transaction'] = pd.to_datetime(df['last_transaction'], errors='coerce')
    print("`last_transaction` temporal fields parsed successfully.")
else:
    print("`last_transaction` column not found; skipping parsing.")

`last_transaction` temporal fields parsed successfully.


### Step 5: Missing Value Imputation
Structural continuity across numeric arrays and categorical boundaries.
- **Medians (Numeric):** Circumvents skew/distortion from banking heavy-tails (large localized balances).
- **Modes (Categorical):** Maintains the natural probability distributions.
- **Dates:** Explicitly skipped.

In [6]:
numeric_cols = df.select_dtypes(include=['number']).columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

imputation_logs = []

# Numeric Imputation
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        val_median = df[col].median()
        df[col] = df[col].fillna(val_median)
        imputation_logs.append(col)

# Categorical Imputation
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        val_mode = df[col].mode()[0]
        df[col] = df[col].fillna(val_mode)
        imputation_logs.append(col)

if imputation_logs:
    print(f"Applied robust imputation logic to {len(imputation_logs)} columns.")
else:
    print("No missing values required imputation.")

Applied robust imputation logic to 4 columns.


### Step 6: Targeted Anomaly Rationalization
Handling impossible constraints securely.
- **Dependents:** Extreme values (< 0 or > 10) are logically treated as faulty data ingestion elements.
- **Age Context:** Elements measuring < 18 are deliberately retained (representative of minor/joint custody account frameworks).

In [7]:
if 'dependents' in df.columns:
    # Capture extreme inputs strictly above 10 or impossibilities below 0
    faulty_mask = (df['dependents'] > 10) | (df['dependents'] < 0)
    fault_count = faulty_mask.sum()
    if fault_count > 0:
        median_dep = df['dependents'].median()
        df.loc[faulty_mask, 'dependents'] = median_dep
        print(f"Handled {fault_count} dependent anomalies using median substitution.")
    else:
        print("No dependent anomalies detected.")

Handled 5 dependent anomalies using median substitution.


### Step 7: Feature Engineering Pipeline
Enriching predictive bounds organically:
- `is_active` → Binary indicator of recent activity via transaction existence.
- `balance_diff` → Momentum measurement (cash flow trajectory).
- `customer_tenure_years` → Time normalization.
- `age_group` → Expanded binning constraints protecting against upper-edge bounds.

In [8]:
engineered_count = 0

if 'last_transaction' in df.columns:
    df['is_active'] = df['last_transaction'].notna().astype(int)
    engineered_count += 1

if 'current_balance' in df.columns and 'previous_month_balance' in df.columns:
    df['balance_diff'] = df['current_balance'] - df['previous_month_balance']
    engineered_count += 1

if 'vintage' in df.columns:
    df['customer_tenure_years'] = (df['vintage'] / 365).round(2)
    engineered_count += 1

if 'age' in df.columns:
    # Extrapolated upper bound strictly preventing max-overflow crashes
    bins = [0, 18, 30, 45, 60, 120]
    labels = ['0-17', '18-29', '30-44', '45-59', '60+']
    df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)
    engineered_count += 1

print(f"Engineered {engineered_count} new features successfully.")

Engineered 4 new features successfully.


### Step 8: Safe Column Pruning & Data Integrity Guardrails
Applying conditional enforcement of data types and clearing non-predictive variables securely.

In [9]:
columns_dropped = []

# City Categorical Safe Handling
if 'city' in df.columns:
    if pd.api.types.is_numeric_dtype(df['city']):
        print("`city` is numeric. Maintained as regional ID structure.")
    else:
        df['city'] = df['city'].astype('category')
        print("`city` securely cast as categorical array.")

# Drop Legacy Features Safely if present
drop_targets = ['customer_id', 'branch_code', 'vintage']
for target in drop_targets:
    if target in df.columns:
        df.drop(columns=[target], inplace=True)
        columns_dropped.append(target)

print(f"Removed {len(columns_dropped)} non-predictive/obsolete arrays: {columns_dropped}")
df_final = df.copy()

`city` is numeric. Maintained as regional ID structure.
Removed 3 non-predictive/obsolete arrays: ['customer_id', 'branch_code', 'vintage']


### Step 9: Defensive Validation Layer
We assert strict compliance checks guaranteeing dataset structure is identical to intended logic frameworks before saving. Bypassing issues is unacceptable.

In [11]:
print("Executing Final Validation Assertions...")

# 1. Duplicate integrity
assert df_final.duplicated().sum() == 0, "Validation Failed: Duplicates persist."
print("Assertion Passed: No duplicates.")

# 2. Missing data structural checks (Only dates should be null)
remaining_nulls = df_final.isnull().sum()
null_cols = remaining_nulls[remaining_nulls > 0].index.tolist()
expected_nulls = ['last_transaction']
assert all(nc in expected_nulls for nc in null_cols), f"Validation Failed: Unauthorized NaNs in {null_cols}"
print("Assertion Passed: Missing value limits enforced.")

print(f"\nFinal shape verification: {df_final.shape}")

Executing Final Validation Assertions...
Assertion Passed: No duplicates.
Assertion Passed: Missing value limits enforced.

Final shape verification: (28382, 22)


### Step 10: Final Export & Summary
Data Quality Summary is formulated and the dataframe is safely exported to the central processing hub.

In [12]:
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
output_path = f'{output_dir}/cleaned_banking_churn.csv'
df_final.to_csv(output_path, index=False)

print("\n=============================================")
print("  PIPELINE DATA QUALITY & EXIT SUMMARY ")
print("=============================================")
print(f"Total Rows Processed: {original_shape[0]} \n  ⮑ Final Output Rows: {df_final.shape[0]}")
print(f"Columns Filtered Out: {len(columns_dropped)}")
print(f"Total Features Engineered: {engineered_count}")
print(f"Production Integrity Validation: PASSED")
print(f"Dataset successfully deployed to: {output_path}")
print("=============================================")


  PIPELINE DATA QUALITY & EXIT SUMMARY 
Total Rows Processed: 28382 
  ⮑ Final Output Rows: 28382
Columns Filtered Out: 3
Total Features Engineered: 4
Production Integrity Validation: PASSED
Dataset successfully deployed to: ../data/processed/cleaned_banking_churn.csv
